# SimOC Demo
**Agent-based simulation of object-centric processes discovered from OCEL 2.0 event logs.**

This notebook runs the full SimOC pipeline on Knopp's Order Management benchmark log in 5 cells.

## Cell 1 — Load Benchmark Log

In [ ]:
import warnings; warnings.filterwarnings("ignore")

from simoc.ingestion import load_ocel, compute_summary

data = load_ocel("data/order-management.sqlite")
print(compute_summary(data))

## Cell 2 — Discover Process Model

In [ ]:
import numpy as np
from simoc.discovery.interaction_graph import discover_interaction_graph
from simoc.discovery.cardinality import compute_spawning_profiles
from simoc.discovery.behavioral import discover_behavioral
from simoc.discovery.patterns import discover_patterns
from simoc.discovery.data_structures import DiscoveryResult

bd, oig, tc = discover_interaction_graph(data)
sp = compute_spawning_profiles(data, bd, tc)
bp = discover_behavioral(data, bd, tc)
ip = discover_patterns(data, bd, tc)
dr = DiscoveryResult(bd, oig, tc, sp)

# --- Type Classification ---
print("=== Type Classification ===")
for t in sorted(tc.classification):
    role = tc.classification[t]
    parent = tc.parent_map.get(t, "—")
    print(f"  {t:12s}  {role:8s}  parent: {parent}")

# --- Spawning ---
print("\n=== Spawning Pairs ===")
for (pt, ct), p in sorted(sp.items()):
    print(f"  {pt} -> {ct}: mean={np.mean(p.raw_counts):.2f}, fit={p.fitted.name}")

# --- Sync Rules ---
print("\n=== Synchronization Rules ===")
for k, r in sorted(ip.synchronization_rules.items()):
    print(f"  {r.activity}, {r.synced_type}: condition={r.condition}, delay_mean={np.mean(r.raw_sync_delays)/3600:.1f}h, n={r.n_instances}")

# --- Binding ---
print(f"\n=== Binding Policies: {len(ip.binding_policies)} ===")
for k, r in ip.binding_policies.items():
    print(f"  {r.source_type} -> {r.target_type} at '{r.binding_activity}', n={r.n_instances}")

# --- Batching ---
print(f"\n=== Batching Rules: {len(ip.batching_rules)} ===")
for k, r in sorted(ip.batching_rules.items()):
    print(f"  {r.activity}, {r.batched_type}: trigger={r.trigger_type}, mean_size={np.mean(r.raw_batch_sizes):.1f}, n={r.n_instances}")

# --- Release ---
print(f"\n=== Release Rules: {len(ip.release_rules)} ===")
for k, r in sorted(ip.release_rules.items()):
    print(f"  ({r.type_1}, {r.type_2}): at '{r.release_activity}', {r.release_condition}")

## Cell 3 — Simulate

In [ ]:
from simoc.simulation.runner import SimulationRunner
from simoc.evaluation._helpers import sim_result_to_oceldata
from collections import Counter

# Match real log duration
duration = (data.events["timestamp"].max() - data.events["timestamp"].min()).total_seconds()
runner = SimulationRunner(dr, bp, ip)
result = runner.run(duration=duration, seed=42)
syn = sim_result_to_oceldata(result)

# Side-by-side comparison
print(f"{'Metric':<30s} {'Real':>10s} {'Synthetic':>10s}")
print("-" * 52)
print(f"{'Events':<30s} {len(data.events):>10d} {len(syn.events):>10d}")
print(f"{'Objects':<30s} {len(data.objects):>10d} {len(syn.objects):>10d}")
print(f"{'O2O relations':<30s} {len(data.o2o):>10d} {len(syn.o2o):>10d}")

real_types = data.objects["object_type"].value_counts().to_dict()
syn_types = syn.objects["object_type"].value_counts().to_dict()
print()
for t in sorted(set(real_types) | set(syn_types)):
    print(f"  {t:<28s} {real_types.get(t, 0):>10d} {syn_types.get(t, 0):>10d}")

real_acts = data.events["activity"].value_counts().to_dict()
syn_acts = syn.events["activity"].value_counts().to_dict()
print()
for a in sorted(set(real_acts) | set(syn_acts)):
    print(f"  {str(a):<28s} {real_acts.get(a, 0):>10d} {syn_acts.get(a, 0):>10d}")

## Cell 4 — Evaluate Against Baselines

In [ ]:
from simoc.evaluation.baselines import (
    build_flat_simod_runner, build_independent_runner, build_random_binding_runner,
)
from simoc.evaluation.experiment import run_experiment, ExperimentConfig

methods = {
    "SimOC": runner,
    "Flat Simod": build_flat_simod_runner(data, dr, bp, case_type="orders"),
    "Independent": build_independent_runner(data, dr, bp),
    "Random Binding": build_random_binding_runner(dr, bp, ip),
}

config = ExperimentConfig(duration=duration, n_runs=5, seeds=list(range(5)))
exp = run_experiment(data, methods, config, sync_rules=ip.synchronization_rules)

print(exp.summary_table().to_string())
print("\n--- Significance Tests (SimOC vs baselines, p < 0.05 = *) ---")
sig = exp.significance_tests("SimOC")
for _, row in sig.iterrows():
    marker = " *" if row["significant"] else ""
    print(f"  {row['baseline']:15s}  {row['metric']:40s}  p={row['p_value']:.4f}{marker}")

## Cell 5 — OC-DFG Visual Comparison

In [ ]:
import matplotlib.pyplot as plt
from simoc.evaluation._helpers import discover_oc_dfg
from collections import defaultdict

real_dfg = discover_oc_dfg(data)
syn_dfg = discover_oc_dfg(syn)

type_colors = {t: c for t, c in zip(sorted(data.object_types),
               ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c"])}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, dfg, title in [(axes[0], real_dfg, "Real Log"), (axes[1], syn_dfg, "SimOC Synthetic")]:
    # Build adjacency by type
    nodes = set()
    for (src, tgt, ot), freq in dfg.items():
        nodes.add(src); nodes.add(tgt)
    nodes = sorted(nodes)

    # Simple circular layout
    n = len(nodes)
    pos = {node: (np.cos(2*np.pi*i/n), np.sin(2*np.pi*i/n)) for i, node in enumerate(nodes)}

    # Draw edges
    max_freq = max(dfg.values()) if dfg else 1
    for (src, tgt, ot), freq in dfg.items():
        if src not in pos or tgt not in pos:
            continue
        x = [pos[src][0], pos[tgt][0]]
        y = [pos[src][1], pos[tgt][1]]
        lw = max(0.5, 3 * freq / max_freq)
        ax.plot(x, y, color=type_colors.get(ot, "gray"), alpha=0.5, linewidth=lw)

    # Draw nodes
    for node in nodes:
        ax.scatter(*pos[node], s=200, c="white", edgecolors="black", zorder=5)
        ax.text(pos[node][0], pos[node][1], node, ha="center", va="center", fontsize=6, zorder=6)

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_aspect("equal"); ax.axis("off")

# Legend
handles = [plt.Line2D([0], [0], color=c, lw=2, label=t) for t, c in type_colors.items() if t in data.object_types]
fig.legend(handles=handles, loc="lower center", ncol=len(handles), fontsize=9)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()